## **Python 데이터베이스 강의**
SQLite3 완전 정복 — 연결부터 CRUD까지

SQLite3는 별도 서버 없이 파일 하나로 동작하는 경량 데이터베이스로 Python 표준 라이브러리에 포함되어 있어 별도 설치가 필요 없습니다.

데이터베이스의 모든 조작은 CRUD 네 가지로 요약됩니다.
<img src="https://media.licdn.com/dms/image/v2/D5612AQHMhCFuMX-R8w/article-cover_image-shrink_720_1280/article-cover_image-shrink_720_1280/0/1714631128138?e=2147483647&v=beta&t=2T8DQHZafA1NHndeBmV_XmyCvZzEg5wi6IhhgbN69Gw" width=800 height=300>

## ① DB 초기화 — init_db()

데이터베이스 파일에 연결하고, 테이블을 생성하는 과정입니다.

sqlite3.connect()는 파일이 없으면 자동으로 생성합니다. cursor는 SQL 명령을 실행하는 객체입니다.


In [1]:
import sqlite3
import os

DB_NAME = "test_sqlite3.db"

def init_db():
    # 이전 파일이 있으면 삭제 (초기화)
    if os.path.exists(DB_NAME):
        os.remove(DB_NAME)

    # 1. DB 연결 — 파일이 없으면 자동 생성
    conn = sqlite3.connect(DB_NAME)

    # 2. Cursor 생성 — SQL 실행 담당 객체
    cursor = conn.cursor()

    # 3. 테이블 생성 (CREATE TABLE)
    cursor.execute("""
        CREATE TABLE users (
            id       INTEGER PRIMARY KEY,
            name     TEXT NOT NULL,
            fullname TEXT
        )
    """)

    conn.commit()  # 변경사항 확정 (필수!)
    return conn

## ② C — Create (데이터 삽입)
executemany()는 리스트 데이터를 한 번에 여러 행으로 삽입합니다.

In [2]:
# Assuming init_db() from cell L-3q6dEoKLbT has been defined and is in scope.
# Re-initialize conn and cursor for the new test_sqlite3.db database.

conn = init_db() # Call init_db to get a fresh connection to test_sqlite3.db
cursor = conn.cursor() # Get a cursor for this new connection

users_data = [
    ('spongebob', 'Spongebob Squarepants'),
    ('patrick',   'Patrick Star'),
    ('squidward', 'Squidward Tentacles'),
]

# executemany: 리스트를 반복하며 여러 행 삽입
cursor.executemany(
    "INSERT INTO users (name, fullname) VALUES (?, ?)",
    users_data
)
conn.commit()
print(f"{len(users_data)}개의 사용자 데이터 삽입 완료.")

3개의 사용자 데이터 삽입 완료.


## ③ R — Read (데이터 조회)

fetchall()은 모든 행을 리스트로, fetchone()은 첫 번째 행 하나만 반환합니다.

결과는 튜플(tuple) 형식이며, 인덱스로 각 컬럼에 접근합니다.


In [3]:
# 전체 조회 — fetchall()
cursor.execute("SELECT id, name, fullname FROM users")
rows = cursor.fetchall()

for row in rows:
    print(f"ID: {row[0]}, Name: {row[1]}, FullName: {row[2]}")


ID: 1, Name: spongebob, FullName: Spongebob Squarepants
ID: 2, Name: patrick, FullName: Patrick Star
ID: 3, Name: squidward, FullName: Squidward Tentacles


In [4]:
# 조건 조회 — WHERE + fetchone()
cursor.execute(
    "SELECT * FROM users WHERE name = ?",
    ('patrick',)   # 튜플 주의: 쉼표 필수
)
patrick = cursor.fetchone()
print(patrick)

(2, 'patrick', 'Patrick Star')


## ④ U — Update (데이터 수정)

UPDATE ... SET ... WHERE 구문으로 특정 행의 값을 수정합니다.

WHERE 조건을 빠뜨리면 테이블 전체가 수정되니 항상 확인하세요.

In [5]:
new_fullname = 'S. Squarepants'

cursor.execute(
    "UPDATE users SET fullname = ? WHERE name = ?",
    (new_fullname, 'spongebob')
)
conn.commit()

# 변경 확인
cursor.execute("SELECT * FROM users WHERE name = 'spongebob'")
print(cursor.fetchone())

(1, 'spongebob', 'S. Squarepants')


## ⑤ D — Delete (데이터 삭제)

DELETE FROM ... WHERE로 조건에 맞는 행을 삭제합니다.

WHERE 없이 실행하면 테이블의 모든 데이터가 삭제됩니다.


In [6]:
cursor.execute(
    "DELETE FROM users WHERE name = ?",
    ('squidward',)
)
conn.commit()

# 최종 데이터 확인
cursor.execute("SELECT * FROM users")
for row in cursor.fetchall():
    print(row)

(1, 'spongebob', 'S. Squarepants')
(2, 'patrick', 'Patrick Star')


## 실습과제 1

- https://github.com/ancestor9/data/blob/main/customers.csv 을 읽어와서 데이터베이스를 만들고 user하는 table에 CRUD하는 코드를 만들어달라


In [7]:
import gradio as gr

def greet(name, intensity):
    return "Hello, " + name + "!" * int(intensity)

demo = gr.Interface(
    fn=greet,
    inputs=["text", "slider"],
    outputs=["text"],
    api_name="predict"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://017b1b39dac5d85461.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import pandas as pd
import sqlite3
import os

CUSTOMER_DB_NAME = "customer_data.db"

def init_customer_db():
    """고객 데이터베이스 파일을 초기화합니다."""
    if os.path.exists(CUSTOMER_DB_NAME):
        os.remove(CUSTOMER_DB_NAME)
        print(f"이전 데이터베이스 파일 '{CUSTOMER_DB_NAME}' 삭제 완료.")

    conn = sqlite3.connect(CUSTOMER_DB_NAME)
    cursor = conn.cursor()

    print("\n--- [CREATE] 'customers' 테이블 생성 ---")
    cursor.execute("""
        CREATE TABLE customers (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT NOT NULL UNIQUE,
            full_name TEXT
        )
    """)
    conn.commit()
    print("테이블 'customers' 생성 완료.")
    return conn

def perform_customer_crud_operations(conn):
    """CRUD 작업을 수행하고 상위 10개 데이터를 확인합니다."""
    cursor = conn.cursor()
    csv_url = "https://raw.githubusercontent.com/ancestor9/data/main/customers.csv"

    # [C] Create - CSV에서 상위 10개 데이터만 추출하여 삽입
    print("\n--- [C]reate: CSV에서 고객 데이터 상위 10개 삽입 ---")
    try:
        df = pd.read_csv(csv_url)
        # 상위 10개만 슬라이싱 (강의 시 데이터 제어 강조)
        df_top10 = df.head(10)

        # '고객ID'를 username과 full_name으로 사용
        customer_data = df_top10.apply(lambda row: (row['고객ID'], row['고객ID']), axis=1).tolist()

        cursor.executemany("INSERT INTO customers (username, full_name) VALUES (?, ?)", customer_data)
        conn.commit()
        print(f"{len(customer_data)}개의 초기 데이터 삽입 완료.")
    except Exception as e:
        print(f"데이터 삽입 중 오류: {e}")
        return

    # [R] Read - 상위 10개 조회 (LIMIT 사용)
    print("\n--- [R]ead: 고객 조회 (상위 10개) ---")
    cursor.execute("SELECT id, username, full_name FROM customers LIMIT 10")
    rows = cursor.fetchall()
    for row in rows:
        print(f"ID: {row[0]}, Username: {row[1]}, FullName: {row[2]}")

    # [U] Update - 특정 고객 정보 수정 (CUST_0002)
    print("\n--- [U]pdate: CUST_0002 고객 정보 수정 ---")
    target_user = 'CUST_0002'
    new_name = 'Updated User 02'
    cursor.execute("UPDATE customers SET full_name = ? WHERE username = ?", (new_name, target_user))
    conn.commit()

    # 수정 확인
    cursor.execute("SELECT * FROM customers WHERE username = ?", (target_user,))
    print(f"수정 확인 -> {cursor.fetchone()}")

    # [D] Delete - 특정 고객 삭제 (CUST_0003)
    print("\n--- [D]elete: CUST_0003 고객 삭제 ---")
    cursor.execute("DELETE FROM customers WHERE username = ?", ('CUST_0003',))
    conn.commit()

    # 삭제 확인 (조회 결과가 None인지 확인)
    cursor.execute("SELECT * FROM customers WHERE username = 'CUST_0003'")
    if cursor.fetchone() is None:
        print("삭제 확인 -> CUST_0003 데이터가 존재하지 않습니다.")

    # [R] 최종 결과 확인 (최종 10개 혹은 남은 데이터 전체)
    print("\n--- [R]ead: 최종 CRUD 작업 반영 결과 (상위 10개) ---")
    cursor.execute("SELECT * FROM customers ORDER BY id ASC LIMIT 10")
    final_rows = cursor.fetchall()

    for row in final_rows:
        print(f"최종 데이터: {row}")

# --- 실행부 ---
if __name__ == "__main__":
    conn = None
    try:
        conn = init_customer_db()
        perform_customer_crud_operations(conn)
    finally:
        if conn:
            conn.close()
            print(f"\n데이터베이스 연결 종료.")

이전 데이터베이스 파일 'customer_data.db' 삭제 완료.

--- [CREATE] 'customers' 테이블 생성 ---
테이블 'customers' 생성 완료.

--- [C]reate: CSV에서 고객 데이터 상위 10개 삽입 ---
10개의 초기 데이터 삽입 완료.

--- [R]ead: 고객 조회 (상위 10개) ---
ID: 1, Username: CUST_0001, FullName: CUST_0001
ID: 2, Username: CUST_0002, FullName: CUST_0002
ID: 3, Username: CUST_0003, FullName: CUST_0003
ID: 4, Username: CUST_0004, FullName: CUST_0004
ID: 5, Username: CUST_0005, FullName: CUST_0005
ID: 6, Username: CUST_0006, FullName: CUST_0006
ID: 7, Username: CUST_0007, FullName: CUST_0007
ID: 8, Username: CUST_0008, FullName: CUST_0008
ID: 9, Username: CUST_0009, FullName: CUST_0009
ID: 10, Username: CUST_0010, FullName: CUST_0010

--- [U]pdate: CUST_0002 고객 정보 수정 ---
수정 확인 -> (2, 'CUST_0002', 'Updated User 02')

--- [D]elete: CUST_0003 고객 삭제 ---
삭제 확인 -> CUST_0003 데이터가 존재하지 않습니다.

--- [R]ead: 최종 CRUD 작업 반영 결과 (상위 10개) ---
최종 데이터: (1, 'CUST_0001', 'CUST_0001')
최종 데이터: (2, 'CUST_0002', 'Updated User 02')
최종 데이터: (4, 'CUST_0004', 'CUST_0004')
최종 데이터: (5,

## 실습과제 2

- vscode에서 init.py, create.py, read.py, update.py, delete.py로 모듈화하여 실해하는 코드로 refactoring하라

###